<a href="https://colab.research.google.com/github/ale66/learn-datascience/blob/main/week-9/Ranking_world_cup/2026_FIFA_World_Cup.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 2026 FIFA World Cup Simulation and Team Rating Analysis

This notebook simulates the 2026 FIFA World Cup using actual group data. It includes:
*   **Team Rating Systems:** Massey, Keener, and Markov.
*   **Simulation Phases:** Group stage advancement and direct elimination knockout rounds.
*   **Objective:** Predict match outcomes and determine a tournament winner.

### 1: Historical data on International Matches - Download + Read Data

The dataset `results.csv` is obtained from the 'International Football Results from 1872 to 2026' Kaggle dataset, which compiles international football match results.

In [1]:
import pandas as pd
import numpy as np

In [2]:
HIST_DATA = "https://raw.githubusercontent.com/ale66/learn-datascience/main/week-9/Ranking_world_cup/results.csv"

In [3]:
df = pd.read_csv(HIST_DATA)

df.head()

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
0,1872-11-30,Scotland,England,0.0,0.0,Friendly,Glasgow,Scotland,False
1,1873-03-08,England,Scotland,4.0,2.0,Friendly,London,England,False
2,1874-03-07,Scotland,England,2.0,1.0,Friendly,Glasgow,Scotland,False
3,1875-03-06,England,Scotland,2.0,2.0,Friendly,London,England,False
4,1876-03-04,Scotland,England,3.0,0.0,Friendly,Glasgow,Scotland,False


In [4]:

df.tail(10)

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
49467,2026-06-26,Cape Verde,Saudi Arabia,NaN,NaN,FIFA World Cup,Houston,United States,True
49468,2026-06-26,Uruguay,Spain,NaN,NaN,FIFA World Cup,Zapopan,Mexico,True
49469,2026-06-26,Norway,France,NaN,NaN,FIFA World Cup,Foxborough,United States,True
49470,2026-06-26,Senegal,Iraq,NaN,NaN,FIFA World Cup,Toronto,Canada,True
49471,2026-06-27,Algeria,Austria,NaN,NaN,FIFA World Cup,Kansas City,United States,True
49472,2026-06-27,Jordan,Argentina,NaN,NaN,FIFA World Cup,Arlington,United States,True
49473,2026-06-27,Colombia,Portugal,NaN,NaN,FIFA World Cup,Miami Gardens,United States,True
49474,2026-06-27,DR Congo,Uzbekistan,NaN,NaN,FIFA World Cup,Atlanta,United States,True
49475,2026-06-27,Panama,England,NaN,NaN,FIFA World Cup,East Rutherford,United States,True
49476,2026-06-27,Croatia,Ghana,NaN,NaN,FIFA World Cup,Philadelphia,United States,True


Take historical data of the last 8 years, which contains two previous world cups

In [5]:
df = df[df["date"] >= "2018-01-01"]

In [6]:
print(df.shape)

df.head()

(8180, 9)


,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
41297,2018-01-02,Iraq,United Arab Emirates,0.0,0.0,Gulf Cup,Kuwait City,Kuwait,True
41298,2018-01-02,Oman,Bahrain,1.0,0.0,Gulf Cup,Kuwait City,Kuwait,True
41299,2018-01-05,Oman,United Arab Emirates,0.0,0.0,Gulf Cup,Kuwait City,Kuwait,True
41300,2018-01-07,Estonia,Sweden,1.0,1.0,Friendly,Abu Dhabi,United Arab Emirates,True
41301,2018-01-11,Denmark,Sweden,0.0,1.0,Friendly,Abu Dhabi,United Arab Emirates,True


## 2026 FIFA World Cup - group stage

This section fetches the 2026 FIFA World Cup group stage data from a public JSON file.

It then processes this data to extract the teams participating in each group, which will be used for the simulation.

In [7]:
import requests

In [8]:
URL = "https://raw.githubusercontent.com/openfootball/worldcup.json/master/2026/worldcup.json"


In [9]:
data = requests.get(URL).json()

data


{'name': 'World Cup 2026',
 'matches': [{'round': 'Matchday 1',
   'date': '2026-06-11',
   'time': '13:00 UTC-6',
   'team1': 'Mexico',
   'team2': 'South Africa',
   'score': {'ft': [2, 0], 'ht': [1, 0]},
   'goals1': [{'name': 'Julián Quiñones', 'minute': '9'},
    {'name': 'Raúl Jiménez', 'minute': '67'}],
   'goals2': [],
   'group': 'Group A',
   'ground': 'Mexico City'},
  {'round': 'Matchday 1',
   'date': '2026-06-11',
   'time': '20:00 UTC-6',
   'team1': 'South Korea',
   'team2': 'Czech Republic',
   'score': {'ft': [2, 1], 'ht': [0, 0]},
   'goals1': [{'name': 'Hwang In-Beom', 'minute': '67'},
    {'name': 'Oh Hyeon-Gyu', 'minute': '80'}],
   'goals2': [{'name': 'Ladislav Krejcí', 'minute': '59'}],
   'group': 'Group A',
   'ground': 'Guadalajara (Zapopan)'},
  {'round': 'Matchday 8',
   'date': '2026-06-18',
   'time': '12:00 UTC-4',
   'team1': 'Czech Republic',
   'team2': 'South Africa',
   'score': {'ft': [1, 1], 'ht': [1, 0]},
   'goals1': [{'name': 'Michal Sadílek',

extract teams and their respective groups

In [10]:
groups = {}

teams = set()

for match in data["matches"]:

    team1 = match.get("team1")
    team2 = match.get("team2")
    group = match.get("group")

    # skip knockout placeholders like W101 etc.
    if not team1 or not team2:
        continue

    # only group stage matches
    if group is None:
        continue

    if "Group" not in group:
        continue

    if group not in groups:
        groups[group] = set()

    groups[group].add(team1)
    groups[group].add(team2)

    teams.add(team1)
    teams.add(team2)


Prepare groups for visualisation.

In [11]:
groups = {k: sorted(list(v)) for k, v in groups.items()}

teams = sorted(list(teams))

sanity check

In [12]:
print("number of teams:", len(teams))
print("number of groups:", len(groups))

for g, t in groups.items():
    print(g, t)

number of teams: 48
number of groups: 12
Group A ['Czech Republic', 'Mexico', 'South Africa', 'South Korea']
Group B ['Bosnia & Herzegovina', 'Canada', 'Qatar', 'Switzerland']
Group C ['Brazil', 'Haiti', 'Morocco', 'Scotland']
Group D ['Australia', 'Paraguay', 'Turkey', 'USA']
Group E ['Curaçao', 'Ecuador', 'Germany', 'Ivory Coast']
Group F ['Japan', 'Netherlands', 'Sweden', 'Tunisia']
Group G ['Belgium', 'Egypt', 'Iran', 'New Zealand']
Group H ['Cape Verde', 'Saudi Arabia', 'Spain', 'Uruguay']
Group I ['France', 'Iraq', 'Norway', 'Senegal']
Group J ['Algeria', 'Argentina', 'Austria', 'Jordan']
Group K ['Colombia', 'DR Congo', 'Portugal', 'Uzbekistan']
Group L ['Croatia', 'England', 'Ghana', 'Panama']


Pick up the corresponding dataset

In [13]:
df_48 = df[
    (df["home_team"].isin(teams)) &
    (df["away_team"].isin(teams))
].copy()

## 2: Massey Ratings (Linear Algebra Version)


Core Formula: $\overline{M}\mathbf{r}=\mathbf{p}$

where

*   $M$ = simple match matrix, always the same;
*   $\overline{M}$ = Massey's *mutilated* matrix;
*   $\mathbf{r}$ = ratings vector, to be solved, and
*   $\mathbf{p}$ = goal difference vector, known.

#### Construct the simple M matrix

1. Teams

In [14]:
n = len(teams)

team_index = {t:i for i,t in enumerate(teams)}


2. Fill Matrix

In [15]:
M = np.zeros((n, n))
p = np.zeros(n)

for _, row in df_48.iterrows():
    home = team_index[row["home_team"]]
    away = team_index[row["away_team"]]

    hs = row["home_score"]
    as_ = row["away_score"]

    if np.isnan(hs) or np.isnan(as_):
        continue

    diff = hs - as_

    # Massey matrix
    M[home, home] += 1
    M[away, away] += 1
    M[home, away] -= 1
    M[away, home] -= 1

    p[home] += diff
    p[away] -= diff

##### Now construct Massey's mutilated matrix $\overline{M}$

In [16]:
M[-1, :] = 1
p[-1] = 0

1. Solve  $\overline{M}\mathbf{r}=\mathbf{p}$ by finding the least-squares solution, thanks to `numpy.linalg.lstsq`.

In [17]:
ratings_massey = np.linalg.lstsq(M, p, rcond=None)[0]

5. Output

In [18]:
massey_df = pd.DataFrame({
    "team": teams,
    "massey": ratings_massey
})

massey_df = massey_df.sort_values("massey", ascending=False)
massey_df.reset_index(drop=True, inplace=True)

massey_df.head(10)

,team,massey
0,France,1.396694
1,Spain,1.305083
2,Brazil,1.288671
3,Belgium,1.278426
4,Portugal,1.263877
5,Argentina,1.172778
6,England,1.013288
7,Netherlands,0.992126
8,Austria,0.930968
9,Uruguay,0.739110


## 3: Keener Ratings (Perron vector)

Core Idea: solve the eigenvalue problem

$A\mathbf{r} = \lambda \mathbf{r}$

to find the dominant (Perron) vector, which will represent the team ratings.

If $A$ is irreducible and non-negative, the Perron-Frobenius theorem guarantees a unique positive eigenvector corresponding to the largest eigenvalue $\lambda$.


In [19]:
A = np.zeros((n, n))


In [20]:
# Filter out rows with NaN scores from df_48 for Keener ratings calculation
df_48_keener = df_48.dropna(subset=["home_score", "away_score"]).copy()

for _, row in df_48_keener.iterrows():
    i = team_index[row["home_team"]]
    j = team_index[row["away_team"]]

    A[i, j] += row["home_score"] + 1

    A[j, i] += row["away_score"] + 1


Normalisation

In [21]:
col_sum = A.sum(axis=0)

# Avoid division by zero for teams that might not have played any match
# Replace 0 with 1 in col_sum where it's 0 to prevent NaN/Inf in A after division
col_sum[col_sum == 0] = 1

A = A / col_sum


##### Calculate Perron's eigenvector

In [22]:
# Ensure A is not all zeros or contains NaNs
# Also handle cases where A might contain inf
if not np.all(A == 0) and not np.any(np.isnan(A)) and not np.any(np.isinf(A)):

    eigvals, eigvecs = np.linalg.eig(A)

    idx = np.argmax(eigvals.real)

    keener_ratings = eigvecs[:, idx].real

    # Key: Standardize (non-negative + normalize)
    keener_ratings = np.abs(keener_ratings)
    keener_ratings = keener_ratings / keener_ratings.sum()

    keener_df = pd.DataFrame({
        "team": teams,
        "rating": keener_ratings
    }).sort_values("rating", ascending=False)

    keener_df.reset_index(drop=True, inplace=True)

    print(keener_df.head(10))
else:
    print("Matrix A is invalid (all zeros, contains NaN, or Inf values). Cannot compute Keener ratings.")

          team    rating
0        Spain  0.046926
1       France  0.044437
2       Brazil  0.043469
3      Croatia  0.039194
4    Argentina  0.037058
5     Colombia  0.035381
6     Portugal  0.035029
7  Netherlands  0.034909
8      Germany  0.033969
9        Japan  0.032634


## 4. Markov Ratings

Markov ratings model the strength of teams based on a Markov chain, where teams are states and the transitions between states are determined by match outcomes (goal scores).

The core idea is to establish a transition matrix where $P_{ij}$ represents the 'strength' or 'likelihood' of team $i$ transitioning to or dominating team $j$.

The stationary distribution of this Markov chain then provides the long-term ratings of each team.

### Construct the Markov Transition Matrix

In [23]:
n = len(teams)


In [24]:
A_markov = np.zeros((n, n))


In [25]:
# Filter out matches with NaN scores
df_48_markov = df_48.dropna(subset=["home_score", "away_score"]).copy()

for _, row in df_48_markov.iterrows():

    home = row["home_team"]
    away = row["away_team"]

    i = team_index[home]
    j = team_index[away]

    hs = row["home_score"]
    aw = row["away_score"]

    # Home team wins
    if hs > aw:
        # Loser (away) gives weight to winner (home)
        A_markov[j, i] += 1

    # Away team wins
    elif aw > hs:
        A_markov[i, j] += 1

    # it's a tie
    else:
        A_markov[i, j] += 0.5
        A_markov[j, i] += 0.5


Handle teams with no outgoing edges (no matches played)

In [26]:
for i in range(n):
    if A_markov[i].sum() == 0:
        A_markov[i] = np.ones(n)


Row normalization

Let's make the transition matrix stochastic: each row sums to 1.

In [27]:
row_sums = A_markov.sum(axis=1)

S = A_markov / row_sums[:, None]

##### Damping/Escape factor

This is the escape probability, to be determined empirically.

We begin with a lowish value of 0.15, which is PageRank, essentially.

Experiment with higher values, up to 0.5, roughly.

In [28]:
ALPHA = 0.15


Create the Escape Matrix

In [29]:
E = np.ones((n, n)) / n


In [30]:
P_markov = (
    (1 - ALPHA) * S + ALPHA * E
)


Solve

In [31]:
eigvals, eigvecs = np.linalg.eig(P_markov.T)

idx = np.argmin(np.abs(eigvals - 1))

ratings = np.real(eigvecs[:, idx])

ratings = ratings / ratings.sum()


Output rankings

In [32]:
markov_df = pd.DataFrame({
    "team": teams,
    "rating": ratings
})

markov_df = markov_df.sort_values(
    "rating",
    ascending=False
)

markov_df.reset_index(drop=True, inplace=True)

print(markov_df.head(10))

          team    rating
0       France  0.051469
1       Brazil  0.046604
2        Spain  0.045516
3    Argentina  0.043150
4     Colombia  0.042032
5      Croatia  0.038555
6     Portugal  0.036330
7      Uruguay  0.032735
8  Netherlands  0.031273
9        Japan  0.030890


### 5. Putting it All Together: 2026 World Cup Simulation

Create a single function `simulate_world_cup` that can be used with any rating system (Massey, Keener, Markov). This function will handle:

1.  **Group Stage Advancement:** Identifying the top two teams from each group and the best eight third-placed teams.
2.  **Knockout Rounds:** Simulating matches based on the provided ratings until a single winner is determined.

#### Simulating Individual Matches

In [33]:
import random

def simulate_match(team1, team2, rating_map):
    rating1 = rating_map.get(team1)
    rating2 = rating_map.get(team2)

    if rating1 is None or rating2 is None:
        raise ValueError(f"Missing rating for {team1} or {team2}")

    if rating1 > rating2:
        return team1
    elif rating2 > rating1:
        return team2
    else:
        return random.choice([team1, team2])

#### Group Stage and Third-Place Advancement

In [34]:
def process_groups(ratings_df, rating_col, groups):
    advancing = {}
    third_place = []

    for g, teams in groups.items():
        df = ratings_df[ratings_df['team'].isin(teams)] \
             .sort_values(rating_col, ascending=False)

        advancing[g] = df['team'].head(2).tolist()

        if len(df) >= 3:
            third_place.append((df.iloc[2]['team'], df.iloc[2][rating_col], g))

    third_place = sorted(third_place, key=lambda x: x[1], reverse=True)[:8]
    third_place = [t[0] for t in third_place]

    return advancing, third_place

#### Knockout Round Progression

In [35]:
def play_round(teams, ratings_df, rating_col, round_name=""):
    if len(teams) % 2:
        winners, pairs = [teams.pop(0)], zip(teams[::2], teams[1::2])
    else:
        winners, pairs = [], zip(teams[::2], teams[1::2])

    matches = []

    # Create a rating_map from the ratings_df for efficient lookup
    rating_map = ratings_df.set_index('team')[rating_col].to_dict()

    for t1, t2 in pairs:
        w = simulate_match(t1, t2, rating_map)
        winners.append(w)
        matches.append({"team1": t1, "team2": t2, "winner": w})

    return winners, {"round": round_name, "matches": matches}

#### `simulate_world_cup` Function: Running the Full Tournament Simulation

This is the main simulation function that orchestrates the entire World Cup. It first processes the group stage to determine the initial set of advancing teams. Then, it iteratively runs knockout rounds until a single champion is determined. It also collects the results of each round in a bracket for later display.

In [36]:
def simulate_world_cup(ratings_df, rating_column_name, groups, simulation_name, initial_round_size=32):
    print(f"\n--- World Cup Simulation ({simulation_name}) ---")

    advancing, third_place = process_groups(ratings_df, rating_column_name, groups)

    current = [t for v in advancing.values() for t in v] + third_place
    random.shuffle(current)

    bracket = []
    round_size = initial_round_size

    while len(current) > 1:
        current, round_result = play_round(
            current,
            ratings_df,
            rating_column_name,
            f"Round of {round_size}"
        )
        bracket.append(round_result)
        round_size //= 2

    print(f"\n🏆 Winner ({simulation_name}): {current[0]}")

    return current[0], bracket

In [37]:
def print_bracket(bracket):
    for r in bracket:
        print(f"\n=== {r['round']} ===")
        for m in r['matches']:
            print(f"{m['team1']} vs {m['team2']} → {m['winner']}")

### Run Simulations with the General Function

In [38]:
massey_winner, bracket = simulate_world_cup(massey_df, 'massey', groups, 'Massey')

print_bracket(bracket)


--- World Cup Simulation (Massey) ---

🏆 Winner (Massey): France

=== Round of 32 ===
Paraguay vs France → France
Bosnia & Herzegovina vs Mexico → Mexico
Spain vs Argentina → Spain
Sweden vs Japan → Japan
Morocco vs Ivory Coast → Morocco
Uruguay vs Norway → Uruguay
South Korea vs Switzerland → Switzerland
Colombia vs USA → Colombia
Senegal vs Iran → Senegal
Croatia vs Portugal → Portugal
Belgium vs Egypt → Belgium
Scotland vs Brazil → Brazil
Netherlands vs Germany → Netherlands
Ecuador vs Canada → Ecuador
Turkey vs England → England
Austria vs Algeria → Austria

=== Round of 16 ===
France vs Mexico → France
Spain vs Japan → Spain
Morocco vs Uruguay → Uruguay
Switzerland vs Colombia → Colombia
Senegal vs Portugal → Portugal
Belgium vs Brazil → Brazil
Netherlands vs Ecuador → Netherlands
England vs Austria → England

=== Round of 8 ===
France vs Spain → France
Uruguay vs Colombia → Uruguay
Portugal vs Brazil → Brazil
Netherlands vs England → England

=== Round of 4 ===
France vs Uruguay

In [39]:
keener_winner, bracket = simulate_world_cup(keener_df, 'rating', groups, 'Keener')

print_bracket(bracket)


--- World Cup Simulation (Keener) ---

🏆 Winner (Keener): Spain

=== Round of 32 ===
Norway vs Scotland → Scotland
Uruguay vs Spain → Spain
Turkey vs Croatia → Croatia
Ecuador vs Algeria → Ecuador
Argentina vs Belgium → Argentina
Egypt vs Portugal → Portugal
Senegal vs Switzerland → Switzerland
Japan vs Netherlands → Netherlands
Morocco vs Qatar → Morocco
Paraguay vs Austria → Paraguay
France vs Germany → France
Colombia vs Australia → Colombia
Czech Republic vs England → England
Mexico vs Saudi Arabia → Mexico
South Korea vs Panama → South Korea
Brazil vs Sweden → Brazil

=== Round of 16 ===
Scotland vs Spain → Spain
Croatia vs Ecuador → Croatia
Argentina vs Portugal → Argentina
Switzerland vs Netherlands → Netherlands
Morocco vs Paraguay → Paraguay
France vs Colombia → France
England vs Mexico → Mexico
South Korea vs Brazil → Brazil

=== Round of 8 ===
Spain vs Croatia → Spain
Argentina vs Netherlands → Argentina
Paraguay vs France → France
Mexico vs Brazil → Brazil

=== Round of 4 

In [40]:
markov_winner, bracket = simulate_world_cup(markov_df, 'rating', groups, 'Markov')

print_bracket(bracket)


--- World Cup Simulation (Markov) ---

🏆 Winner (Markov): France

=== Round of 32 ===
Australia vs Switzerland → Switzerland
Iran vs Brazil → Brazil
Austria vs Egypt → Egypt
Portugal vs Belgium → Portugal
Scotland vs Ecuador → Ecuador
Japan vs Morocco → Japan
Spain vs Canada → Spain
Ivory Coast vs Tunisia → Tunisia
Senegal vs England → England
Netherlands vs Colombia → Colombia
Mexico vs Saudi Arabia → Mexico
Turkey vs Qatar → Turkey
Germany vs Uruguay → Uruguay
South Korea vs Paraguay → South Korea
Argentina vs Croatia → Argentina
Algeria vs France → France

=== Round of 16 ===
Switzerland vs Brazil → Brazil
Egypt vs Portugal → Portugal
Ecuador vs Japan → Japan
Spain vs Tunisia → Spain
England vs Colombia → Colombia
Mexico vs Turkey → Mexico
Uruguay vs South Korea → Uruguay
Argentina vs France → France

=== Round of 8 ===
Brazil vs Portugal → Brazil
Japan vs Spain → Spain
Colombia vs Mexico → Colombia
Uruguay vs France → France

=== Round of 4 ===
Brazil vs Spain → Brazil
Colombia vs

### Comparative Summary of Top Teams by Rating Method

In [41]:
comparative_top_teams = pd.DataFrame({
    'Massey Ratings': massey_df.head(5)['team'].tolist(),
    'Keener Ratings': keener_df.head(5)['team'].tolist(),
    'Markov Ratings': markov_df.head(5)['team'].tolist()
})

print(f"Massey Winner: {massey_winner}")
print(f"Keener Winner: {keener_winner}")
print(f"Markov Winner: {markov_winner}")

display(comparative_top_teams)

Massey Winner: France
Keener Winner: Spain
Markov Winner: France


,Massey Ratings,Keener Ratings,Markov Ratings
0,France,Spain,France
1,Spain,France,Brazil
2,Brazil,Brazil,Spain
3,Belgium,Croatia,Argentina
4,Portugal,Argentina,Colombia
